# Domain 3 · Claude Code Configuration & Workflows (20%)
## One notebook — Task Statements 3.1 → 3.6

**How to use this notebook.** Domain 3 is mostly *configuration and workflow* — the lesson
is which **real file** to create, which **CLI flag** to run, and when to pick **plan mode**.
So each section either (a) **parses your REAL `.claude/` artifacts** in Exercise 2 / Exercise 5
and makes the rule observable, or (b) makes a **real Claude call** where the task statement is
genuinely a Claude *decision* (§3.4 plan-vs-direct, §3.5 prose-vs-examples, §3.6 structured CI
output). Read the guide quote → the plain-English table → **run the cell** → read the
anti-patterns → open your own file.

The spine: §3.1–3.4 configure the **EX2 team repo** (`exercises/02-claude-code-team/`);
§3.5–3.6 wire its review into **CI** (`exercises/05-cicd-review/`). Same artifacts you built.

> Setup: uses the `anthropic` SDK + your `ANTHROPIC_API_KEY` from the repo-root `.env`.
> Run from `ccaf-prep/notebooks/` (or anywhere under the repo — the setup cell finds the files).

---
## Setup — load the key and locate your real `.claude/` artifacts

This cell loads the key from your single `.env` and resolves the **real** EX2/EX5 folders so
every config section below reads your actual files (never a paraphrase).

In [ ]:
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv
from pathlib import Path, PurePosixPath
import os, json, re

# Portable .env discovery — nearest .env walking up from the working dir (repo root or
# ccaf-prep/), then the sibling course .env. No machine-specific paths.
_candidates = [Path.cwd().parents[1] / "claude-with-anthropic-api" / ".env"]
_found = find_dotenv(filename=".env", usecwd=True)
_envfile = Path(_found) if _found else next((p for p in _candidates if p.exists()), None)
if _envfile: load_dotenv(_envfile); print(f"Loaded .env from: {_envfile}")
else: print("WARNING: no .env found — copy .env.example to .env at the repo root")
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — see .env.example"

client = Anthropic()
MODEL = os.environ.get("CLAUDE_MODEL") or "claude-haiku-4-5"   # cheap default; teaches mechanism
print("Using model:", MODEL)

# Locate the REAL exercise folders regardless of where the kernel's cwd is.
def _find(rel):
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / rel, base / "ccaf-prep" / rel):
            if cand.exists(): return cand
    raise FileNotFoundError(rel + " — run from inside the repo")

EX2 = _find("exercises/02-claude-code-team")     # the team-repo .claude/ config (D3.1-3.4)
EX5 = _find("exercises/05-cicd-review")           # the CI review pipeline (D3.5-3.6)
print("EX2:", EX2)
print("EX5:", EX5)

# A tiny frontmatter reader reused by §3.2 and §3.3 (good enough for these files).
def parse_frontmatter(text):
    m = re.match(r"^---\n(.*?)\n---\n?", text, re.S)
    if not m: return {}, text
    fm, body = {}, text[m.end():]
    for line in m.group(1).splitlines():
        if ":" in line and not line.startswith((" ", "#", "-")):
            k, _, v = line.partition(":")
            fm[k.strip()] = v.strip()
    return fm, body
print("setup OK")

## 3.1 · CLAUDE.md hierarchy, scoping & modular organization

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.1: Configure CLAUDE.md files with appropriate hierarchy, scoping, and modular organization**
>
> **Knowledge of:**
> - The CLAUDE.md configuration hierarchy: user-level (`~/.claude/CLAUDE.md`),
>   project-level (`.claude/CLAUDE.md` or root CLAUDE.md), and directory-level
>   (subdirectory CLAUDE.md files)
> - That user-level settings apply only to that user—instructions in `~/.claude/CLAUDE.md`
>   are not shared with teammates via version control
> - The @import syntax for referencing external files to keep CLAUDE.md modular (e.g.,
>   importing specific standards files relevant to each package)
> - `.claude/rules/` directory for organizing topic-specific rule files as an alternative to a
>   monolithic CLAUDE.md
>
> **Skills in:**
> - Diagnosing configuration hierarchy issues (e.g., a new team member not receiving
>   instructions because they're in user-level rather than project-level configuration)
> - Using @import to selectively include relevant standards files in each package's
>   CLAUDE.md based on maintainer domain knowledge
> - Splitting large CLAUDE.md files into focused topic-specific files in `.claude/rules/` (e.g.,
>   testing.md, api-conventions.md, deployment.md)
> - Using the /memory command to verify which memory files are loaded and diagnose
>   inconsistent behavior across sessions

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| 3-layer hierarchy | Three places a `CLAUDE.md` can live: **user** (`~/.claude/`), **project** (committed in the repo), **directory** (a subfolder). They stack; the more-local layer refines the broader one. |
| user-level not shared | `~/.claude/CLAUDE.md` is **your machine only**. Git never sees it, so teammates never get it. Personal prefs only. |
| **when each layer loads** | Two different triggers. **Eager at launch:** user-level + every `CLAUDE.md` from your cwd *up* to the repo root. **On-demand:** a **directory** `CLAUDE.md` *below* your cwd loads the first time Claude **reads or edits a file inside that subtree** — *not* based on where you launched the CLI. So launching from the repo root does **not** pre-load `src/payments/CLAUDE.md`; touching a file under `src/payments/` does. |
| `@import` | A line `@import ./team-conventions.md` pulls another file *in*, so `CLAUDE.md` stays a thin index instead of one giant file. It is **eager**: the imported file is expanded into context **at launch**, exactly as if you'd inlined it — *not* a conditional "read this only if you need it". (Recursive up to 4 hops.) **Conditional** loading is `.claude/rules/` with `paths:` globs (§3.3), a different mechanism — don't conflate the two. |
| `.claude/rules/` | Topic-split alternative to a monolith: `testing.md`, `deployment.md`, each loaded on its own terms (see §3.3 for path-scoped loading). |
| Diagnose hierarchy bug | "New teammate isn't getting the rules" ⇒ the rules are in **user-level** (not shared) instead of **project-level**. Move them down a layer. |
| `/memory` | The Claude Code command that **lists which CLAUDE.md files are currently loaded** — your debugger for "why is behavior inconsistent across sessions?" |

### 3 · Run it — resolve the hierarchy & the `@import` on your REAL EX2/CLAUDE.md

No model call here: the lesson is *which file loads and who shares it* — a deterministic
property of the files you wrote. This cell reads your actual `EX2/CLAUDE.md`, prints the
precedence order, and **follows its real `@import` line** to the module it pulls in.

In [ ]:
claude_md = (EX2 / "CLAUDE.md").read_text()

# (a) The three layers, in load order. Each more-local layer refines the broader one;
#     ONLY the project + directory layers travel through version control to teammates.
HIERARCHY = [
    ("user",      "~/.claude/CLAUDE.md",                    False, "your machine only"),
    ("project",   "<repo>/CLAUDE.md  or  .claude/CLAUDE.md", True,  "committed -> every teammate"),
    ("directory", "<subdir>/CLAUDE.md",                     True,  "loads only inside that subtree"),
]
print("CLAUDE.md hierarchy (load order — more local refines broader):")
for scope, path, shared, note in HIERARCHY:
    print(f"   {scope:9} {path:42} shared-via-VCS={str(shared):5}  {note}")

# (b) Follow the REAL @import in EX2/CLAUDE.md -> proves modular composition.
imports = re.findall(r"^@import\s+(\S+)", claude_md, re.M)
print("\n@import lines in EX2/CLAUDE.md:", imports)
for rel in imports:
    mod = (EX2 / rel).resolve()
    print(f"   -> {rel}  resolves to  {mod.name}  ({mod.stat().st_size} bytes), merged into the PROJECT layer")

# (c) The diagnose-this skill, made concrete: where do TEAM rules belong?
shared_layers = [s for s, _p, sh, _n in HIERARCHY if sh]
print(f"\nDiagnose 'new teammate missing the rules': team rules MUST live in a SHARED layer "
      f"{shared_layers} — never user-level (it never enters git).")
print("Verify live with the /memory command inside a Claude Code session.")

### 3b · Try it live in your terminal — `/memory` from two locations

The cell above resolves the hierarchy *statically* from the files. To **see the load triggers
fire for real**, run Claude Code yourself and use `/memory` (the official "what's loaded right
now?" command). Three quick checks, ~2 minutes — each proves one of the rules above:

**Check 1 — launch from the repo root → the subdir layer is NOT pre-loaded (eager-up only).**
```bash
cd "$(git rev-parse --show-toplevel)"        # repo root
claude
```
Inside the session, type `/memory`. You should see the **root `CLAUDE.md`** and your
**`~/.claude/CLAUDE.md`** (loaded eagerly, cwd-and-up), but **not** EX2's `CLAUDE.md` or its
`.claude/rules/tests.md` — they live *below* you and haven't been touched yet. Type `exit` (or
Ctrl-D) to quit.

**Check 2 — launch from inside EX2 → its `CLAUDE.md` loads, with the `@import` already expanded (eager).**
```bash
cd "$(git rev-parse --show-toplevel)/ccaf-prep/exercises/02-claude-code-team"
claude
```
Run `/memory`. Now EX2's `CLAUDE.md` appears — and the contents of **`team-conventions.md`** are
already inlined into it (proving `@import` is eager, not conditional). The root `CLAUDE.md` is
still there too (it's a parent of your cwd).

**Check 3 — on-demand directory load + conditional `.claude/rules/`.** Still inside EX2, ask
Claude to touch a matching file, e.g.:
```
> create a file src/widget/Button.test.tsx with a trivial passing test
```
The act of writing a file under that subtree is what would pull in any directory-level
`CLAUDE.md` there, and the `**/*.test.*` glob in `.claude/rules/tests.md` is what makes the **test
convention** load *only* for this file type (§3.3). Re-run `/memory` to confirm the rule is now in
play. (Delete the throwaway file afterward.)

> Out of scope on purpose: launching from a directory **outside** this repo adds nothing new to
> the lesson — the root-vs-subdir contrast in Checks 1–2 is the whole demonstration. Going
> further would be overengineering.

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1: put TEAM conventions in ~/.claude/CLAUDE.md so "they're always on for me".
#   ~/.claude/CLAUDE.md is per-machine and NEVER committed -> teammates never receive them.
#   (This is the exact "new teammate isn't getting the rules" bug in the guide.)

# ANTI-PATTERN 2: one monolithic 800-line CLAUDE.md with every standard inlined.
#   Unmaintainable; every package inherits everything. Use @import modules + .claude/rules/.

# ANTI-PATTERN 3: assume a deep subdirectory CLAUDE.md applies everywhere.
#   It loads ONLY once Claude reads/edits a file inside that subtree (NOT based on where you
#   launched the CLI) -> spread-out conventions (e.g. tests everywhere) need PATH rules instead
#   (see 3.3), not a buried CLAUDE.md.

# ANTI-PATTERN 4: reach for @import expecting CONDITIONAL ("load only if needed") savings.
#   @import is EAGER -> the module is pulled into context at launch, same as inlining. It buys
#   modularity, NOT lazy loading. For load-only-on-match use .claude/rules/ paths: globs (3.3).

print("Correct: TEAM rules in PROJECT CLAUDE.md (committed), kept modular via @import (eager) + "
      ".claude/rules/ (conditional); user-level holds ONLY personal prefs; /memory shows what's loaded.")

### 6 · Self-check (cover the answers)

1. A teammate reports Claude ignores your team's commit-message convention. Where are the rules *probably* living, and where should they go?
2. What does `@import ./team-conventions.md` buy you over inlining that text?
3. True/False: a `CLAUDE.md` in `src/payments/` is loaded for every file in the repo.
4. Which command shows which memory files are currently loaded?

<details><summary>answers</summary>

1. In someone's **user-level** `~/.claude/CLAUDE.md` (not shared via VCS). Move them to the
   **project** `CLAUDE.md` (committed). 2. **Modularity** — `CLAUDE.md` stays a thin index and
   each package imports only the standards it needs. (It does *not* save context: `@import` is
   **eager**, expanded at launch just like inlining — conditional loading is `.claude/rules/`,
   §3.3.) 3. **False** — directory-level loads only when Claude reads/edits a file inside that
   subtree, not from where you launched the CLI. 4. **`/memory`**.
</details></cell id="cell-10">


### 6 · Self-check (cover the answers)

1. A teammate reports Claude ignores your team's commit-message convention. Where are the rules *probably* living, and where should they go?
2. What does `@import ./team-conventions.md` buy you over inlining that text?
3. True/False: a `CLAUDE.md` in `src/payments/` is loaded for every file in the repo.
4. Which command shows which memory files are currently loaded?

<details><summary>answers</summary>

1. In someone's **user-level** `~/.claude/CLAUDE.md` (not shared via VCS). Move them to the
   **project** `CLAUDE.md` (committed). 2. Modularity — `CLAUDE.md` stays a thin index;
   each package imports only the standards it needs. 3. **False** — directory-level loads only
   when working inside that subtree. 4. **`/memory`**.
</details>

## 3.2 · Custom slash commands and skills

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.2: Create and configure custom slash commands and skills**
>
> **Knowledge of:**
> - Project-scoped commands in `.claude/commands/` (shared via version control) vs
>   user-scoped commands in `~/.claude/commands/` (personal)
> - Skills in `.claude/skills/` with SKILL.md files that support frontmatter configuration
>   including `context: fork`, `allowed-tools`, and `argument-hint`
> - The `context: fork` frontmatter option for running skills in an isolated sub-agent
>   context, preventing skill outputs from polluting the main conversation
> - Personal skill customization: creating personal variants in `~/.claude/skills/` with
>   different names to avoid affecting teammates
>
> **Skills in:**
> - Creating project-scoped slash commands in `.claude/commands/` for team-wide
>   availability via version control
> - Using `context: fork` to isolate skills that produce verbose output (e.g., codebase
>   analysis) or exploratory context (e.g., brainstorming alternatives) from the main session
> - Configuring `allowed-tools` in skill frontmatter to restrict tool access during skill
>   execution (e.g., limiting to file write operations to prevent destructive actions)
> - Using `argument-hint` frontmatter to prompt developers for required parameters when
>   they invoke the skill without arguments
> - Choosing between skills (on-demand invocation for task-specific workflows) and
>   CLAUDE.md (always-loaded universal standards)

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| commands: project vs user | `.claude/commands/foo.md` → `/foo`, **committed**, the whole team gets it. `~/.claude/commands/` → personal, **not** shared. |
| SKILL.md frontmatter | A skill is a folder with `SKILL.md`; its YAML frontmatter configures `context`, `allowed-tools`, `argument-hint`. |
| `context: fork` | Run the skill in an **isolated sub-agent**. Its noisy output (a long `git log`, brainstorming) stays out of the main conversation — only the final summary returns. |
| `allowed-tools` | A **least-privilege gate** on the skill: it can only touch the listed tools (e.g. git-reads + file-writes), even if the main session can do more. |
| `argument-hint` | Text shown to prompt the developer for a parameter when they invoke the skill with no args. |
| personal variant | Copy a team skill into `~/.claude/skills/` under a **different name** so you don't shadow the shared one for teammates. |
| skills vs CLAUDE.md | **Skill** = invoked *on demand* for one workflow. **CLAUDE.md** = *always loaded* universal standards. Workflow → skill; standard → CLAUDE.md. |

### 3 · Run it — parse the REAL frontmatter of your EX2 command + skill

Config task: the lesson is *which fields exist and what they scope*. This reads your actual
`/review-pr` command and `changelog` skill and prints the frontmatter that does the work.

### 3b · Try it live — invoke the command, then watch the skill fork

The cell above only *parses* the frontmatter. What those fields actually **do** is only visible
when you run them in a real session — that's the part static parsing can't show.

```bash
cd "$(git rev-parse --show-toplevel)/ccaf-prep/exercises/02-claude-code-team"
claude
```

- **`/review-pr`** (or `/review-pr develop`) → runs your **project-scoped** command. Type `/` and
  confirm it autocompletes — it exists for you *because* it lives in `.claude/commands/` (committed).
  A teammate who cloned the repo gets the exact same command. The same file in `~/.claude/commands/`
  would be personal-only and **gone after their clone** — that's the scope lesson, made observable.
- **Ask "update the changelog to 1.4.0"** (or `/changelog 1.4.0`) → invokes the skill. Its
  frontmatter has **`context: fork`**, so the noisy `git log` walk runs in an **isolated sub-agent**
  and only the final CHANGELOG section returns to your thread. Run **`/context`** before and after to
  watch the main thread stay small — the verbose work never lands in it.
- **`allowed-tools`** (`Bash(git log:*), Bash(git tag:*), Read, Edit, Write`) is a least-privilege
  gate: ask the skill to do something outside that list (e.g. `rm` a file) and it **can't**, even
  though your main session might be allowed to.

> The static cell tells you the fields exist; this tells you `context: fork` isolates and
> `allowed-tools` restricts — the two things the exam actually tests about skills.

In [ ]:
# (a) The project-scoped slash command — lives in .claude/commands/ => shared via VCS.
cmd_text = (EX2 / ".claude/commands/review-pr.md").read_text()
cfm, _ = parse_frontmatter(cmd_text)
print("/review-pr  (.claude/commands/  ->  PROJECT-scoped, committed, every dev gets it)")
for k in ("description", "argument-hint", "allowed-tools"):
    print(f"   {k:13} = {cfm.get(k)!r}")

# (b) The skill — frontmatter fields are the exam's whole point.
skill_text = (EX2 / ".claude/skills/changelog/SKILL.md").read_text()
sfm, _ = parse_frontmatter(skill_text)
print("\nchangelog SKILL.md  (.claude/skills/  ->  on-demand workflow, NOT always-loaded)")
for k in ("name", "allowed-tools", "argument-hint", "context"):
    print(f"   {k:13} = {sfm.get(k)!r}")
forked = sfm.get("context") == "fork"
print(f"   context: fork active? {forked}  -> "
      f"{'verbose git output runs in an ISOLATED sub-agent; only the summary returns' if forked else 'runs inline'}")

# (c) The scope cheat-sheet the exam tests (sample Q4).
print("\nSCOPE:  .claude/commands/  &  .claude/skills/  = team (VCS)   |   "
      "~/.claude/...  = personal (not shared)")
print("PICK:   workflow on demand -> SKILL/command   |   always-on standard -> CLAUDE.md")

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1 (sample Q4 distractor D): define commands in a .claude/config.json "commands" array.
#   No such mechanism exists. A slash command is a MARKDOWN FILE in .claude/commands/.

# ANTI-PATTERN 2 (sample Q4 distractor B): put a team command in ~/.claude/commands/.
#   Personal scope -> NOT shared via VCS; teammates won't have /review-pr after a clone.

# ANTI-PATTERN 3: let a destructive skill run with the session's full tool access.
#   Omitting allowed-tools means it can do anything. Scope it (git-reads + writes only).

# ANTI-PATTERN 4: put an always-on universal standard in a skill (loads only on demand)
#   or a one-off verbose workflow in CLAUDE.md (pollutes every turn). Swap them.

print("Correct: project command/skill -> .claude/commands/ or .claude/skills/ (VCS, markdown + "
      "frontmatter); restrict with allowed-tools; isolate verbose ones with context: fork.")

### 5 · In your own code — you already wrote this

- **`exercises/02-claude-code-team/.claude/commands/review-pr.md:1-5`** — the `/review-pr`
  command frontmatter (`description`, `argument-hint`, `allowed-tools`).
- **`exercises/02-claude-code-team/.claude/skills/changelog/SKILL.md:1-7`** — the skill
  frontmatter with **all three** tested fields: `allowed-tools`, `argument-hint`, `context: fork`.

Open both and confirm each frontmatter key maps to a bullet above.

### 6 · Self-check (cover the answers)

1. You want `/release-notes` available to everyone after a `git pull`. Which directory?
2. A skill does a noisy 500-line codebase scan. Which frontmatter keeps that out of the main chat?
3. What does `allowed-tools` in a skill's frontmatter restrict, and why is that least-privilege?
4. When do you reach for a **skill** vs a **CLAUDE.md** entry?

<details><summary>answers</summary>

1. **`.claude/commands/`** (project-scoped, committed — *not* `~/.claude/commands/`, *not* a
   `config.json` array). 2. **`context: fork`** (isolated sub-agent; only the summary returns).
3. The set of tools the skill may call during execution — it can't perform actions outside that
   list even if the main session could. 4. **Skill** = on-demand task workflow; **CLAUDE.md** =
   always-loaded universal standard.
</details>

## 3.3 · Path-specific rules for conditional convention loading

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.3: Apply path-specific rules for conditional convention loading**
>
> **Knowledge of:**
> - `.claude/rules/` files with YAML frontmatter `paths` fields containing glob patterns for
>   conditional rule activation
> - How path-scoped rules load only when editing matching files, reducing irrelevant context
>   and token usage
> - The advantage of glob-pattern rules over directory-level CLAUDE.md files for conventions
>   that span multiple directories (e.g., test files spread throughout a codebase)
>
> **Skills in:**
> - Creating `.claude/rules/` files with YAML frontmatter path scoping (e.g., `paths:
>   ["terraform/**/*"]`) so rules load only when editing matching files
> - Using glob patterns in path-specific rules to apply conventions to files by type regardless
>   of directory location (e.g., `**/*.test.tsx` for all test files)
> - Choosing path-specific rules over subdirectory CLAUDE.md files when conventions must
>   apply to files spread across the codebase

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| `paths:` glob frontmatter | A `.claude/rules/*.md` file carries `paths: [glob, ...]`. The rule activates **only when the file being edited matches a glob.** |
| loads only on match | Editing `Button.tsx`? The test rule stays silent. Editing `Button.test.tsx`? It loads. Less irrelevant context, fewer tokens. |
| glob beats subdir CLAUDE.md | A directory `CLAUDE.md` covers **one folder**. Test files are scattered **everywhere** — only a `**/*.test.*` glob follows the file *type* across the whole tree. |

### 3b · Try it live — the rule loads for one file, not its sibling

The cell above *simulates* the glob match in Python. To watch the **real** conditional load — where
the harness, not you, decides — run Claude Code inside EX2 and contrast a matching vs a
non-matching file. That asymmetry is the whole payoff of this section:

```bash
cd "$(git rev-parse --show-toplevel)/ccaf-prep/exercises/02-claude-code-team"
claude
```

- Ask it to edit a **non-matching** file: *"add a comment to `src/widget/Button.tsx`"*. Run
  **`/memory`** → `.claude/rules/tests.md` is **not** loaded.
- Now a **matching** one: *"add a case to `src/widget/Button.test.tsx`"*. Run **`/memory`** →
  `tests.md` **is** in context now; its `**/*.test.*` glob fired and the test convention rode in
  with the file *type* — no per-folder config, anywhere in the tree.

That is exactly what a subdirectory `CLAUDE.md` can't do (it's bound to one folder). Delete the
throwaway files afterward.

> Related but different angle: §3.1's §3b *Check 3* triggers this same rule while demonstrating
> the **hierarchy**; here the lens is the **match-vs-no-match** discrimination the exam tests.

### 3 · Run it — read the REAL globs, then watch which files trigger the rule

Rule loading is deterministic (the harness, not the model, matches the glob), so this cell is
pure Python — exactly like §1.4's isolated gate. It reads the **real** `paths:` from your
`tests.md`, then uses real glob matching (`PurePath.full_match`) to show which candidate files
would load it. *That* asymmetry — scattered test files all match, siblings don't — is the lesson.

In [ ]:
rule_text = (EX2 / ".claude/rules/tests.md").read_text()

# (a) Extract the REAL paths: globs from the rule's YAML frontmatter (a multi-line list).
m = re.search(r"paths:\s*\n((?:[ \t]+-\s*.+\n)+)", rule_text)  # indented list only (skip the --- fence)
globs = re.findall(r'-\s*"?([^"\n]+?)"?\s*$', m.group(1), re.M) if m else []
print("Real globs in .claude/rules/tests.md:", globs)

# (b) Candidate files spread across the tree. Which ones LOAD the rule?
candidates = [
    "src/components/Button.test.tsx",   # test, deep in components/
    "api/handlers/users.spec.js",       # spec, in a totally different folder
    "src/payments/tests/fixtures.py",   # under a tests/ dir
    "src/components/Button.tsx",         # sibling of a test file -> must NOT load
    "README.md",                         # unrelated
]
print("\nWhich files trigger the test convention rule? (loads only on a glob match)")
for c in candidates:
    p = PurePosixPath(c)
    hits = [g for g in globs if p.full_match(g)]
    verdict = f"LOAD  (matched {hits})" if hits else "skip  (no match -> rule not loaded)"
    print(f"   {c:34} -> {verdict}")

print("\nThe point: ONE glob rule rides with the file TYPE across every directory — a subdir "
      "CLAUDE.md could only cover one folder and would miss the scattered test files.")

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1 (sample Q6 distractor D): drop a CLAUDE.md in each folder that has tests.
#   Tests are spread EVERYWHERE; you'd need dozens, and new folders silently miss the rule.
#   A directory CLAUDE.md is bound to its directory; it can't follow a file TYPE.

# ANTI-PATTERN 2 (sample Q6 distractor B): pile every convention into the root CLAUDE.md
#   under headings and hope Claude infers which applies. Relies on inference, not matching.

# ANTI-PATTERN 3 (sample Q6 distractor C): make a SKILL per code type for conventions.
#   Skills are on-demand (manual/Claude-chosen) -> NOT the deterministic, automatic,
#   path-triggered loading the scenario requires.

print("Correct: .claude/rules/ file with paths: globs (e.g. **/*.test.*) -> loads automatically "
      "and ONLY when editing a matching file, wherever in the tree it lives (sample Q6 = A).")

### 5 · In your own code — you already wrote this

- **`exercises/02-claude-code-team/.claude/rules/tests.md:1-9`** — the real `paths:` frontmatter
  (`**/*.test.*`, `**/*.spec.*`, `**/tests/**`); line 11 is the convention body that loads only
  on a match.

Open it and note: the rule says nothing about *which directory* — it follows the file **type**.

### 6 · Self-check (cover the answers)

1. Test files sit next to the code they test (`Button.test.tsx` beside `Button.tsx`), all over the repo. Most maintainable way to apply one test convention?
2. Why can't a subdirectory `CLAUDE.md` solve #1 cleanly?
3. When you edit `src/api/handler.ts` (no glob match), does `tests.md` load? What's the token benefit?

<details><summary>answers</summary>

1. A **`.claude/rules/` file with `paths:` globs** like `**/*.test.*` (sample Q6 = A). 2. A
   directory `CLAUDE.md` is bound to one folder; scattered test files would need a copy in every
   folder and new ones get missed. 3. **No** — it loads only on a match, so unrelated edits don't
   pay its token cost.
</details>

## 3.4 · Plan mode vs direct execution

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.4: Determine when to use plan mode vs direct execution**
>
> **Knowledge of:**
> - Plan mode is designed for complex tasks involving large-scale changes, multiple valid
>   approaches, architectural decisions, and multi-file modifications
> - Direct execution is appropriate for simple, well-scoped changes (e.g., adding a single
>   validation check to one function)
> - Plan mode enables safe codebase exploration and design before committing to changes,
>   preventing costly rework
> - The Explore subagent for isolating verbose discovery output and returning summaries to
>   preserve main conversation context
>
> **Skills in:**
> - Selecting plan mode for tasks with architectural implications (e.g., microservice
>   restructuring, library migrations affecting 45+ files, choosing between integration
>   approaches with different infrastructure requirements)
> - Selecting direct execution for well-understood changes with clear scope (e.g., a single-file
>   bug fix with a clear stack trace, adding a date validation conditional)
> - Using the Explore subagent for verbose discovery phases to prevent context window
>   exhaustion during multi-phase tasks
> - Combining plan mode for investigation with direct execution for implementation (e.g.,
>   planning a library migration, then executing the planned approach)

### 3b · Try it live — feel plan mode (the API call can't show this)

The cell above had Claude *classify* tasks via the API. But plan mode itself is a Claude Code
**UX** you can only feel in the CLI — there's no API call that reproduces it. Run both sides:

```bash
cd "$(git rev-parse --show-toplevel)"   # any repo is fine
claude
```

- Press **Shift+Tab** to cycle into **plan mode** (the footer shows it's on). Give it an
  architectural task — *"split this notebook tooling into smaller modules"* — and watch it
  **explore + propose a plan and stop**, making **no edits** until you approve. That pause *is* the
  rework-prevention this section is about.
- **Shift+Tab** back out and give a well-scoped task — *"rename `tmp` to `buffer` in that
  function"* — it just does it. Direct execution; no plan ceremony needed.
- For a verbose discovery phase, ask it to *"use the Explore subagent to map the dependencies"* and
  note a **summary** returns without flooding your main context (the Explore-subagent bullet).

The decision criteria are the same ones Claude applied in the cell above — now you've *run* both
sides instead of only reading the classification.

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| plan mode = complex | Large-scale, **multi-file**, architectural, or **multiple valid approaches** → plan first (explore + design, no edits yet). |
| direct = well-scoped | One clear change with known scope (single-file fix, add a validation) → just do it. |
| plan prevents rework | Exploring and designing *before* editing avoids discovering a wrong assumption after 40 files changed. |
| Explore subagent | A sub-agent that does the **verbose discovery** and returns a *summary*, so the main context isn't flooded. |
| combine the two | Real flow: **plan** the migration, then **direct-execute** the agreed steps. |

### 3 · Run it — let Claude apply the criteria (a REAL decision, not a Python if/switch)

"Determine when to use plan vs direct" is a **judgment** — so the observable must be a real
model call applying the guide's criteria, not a hardcoded router. We hand Claude five tasks and
the criteria, and **Claude** classifies each. Watch the architectural/multi-file ones land on
`plan` and the single-scoped ones on `direct`.

In [ ]:
TASKS = [
    "Restructure our monolithic app into microservices: ~45 files, decisions about service "
    "boundaries and module dependencies.",
    "Add a single null-check to parseDate(); we have the exact failing stack trace.",
    "Migrate from library A to library B across the codebase; two integration approaches exist "
    "with different infrastructure needs.",
    "Rename the local variable `tmp` to `buffer` in one function.",
    "Add a date-range validation conditional to one well-understood endpoint handler.",
]

CRITERIA = ("Classify each task as PLAN mode or DIRECT execution.\n"
            "PLAN: large-scale changes, multiple valid approaches, architectural decisions, "
            "multi-file modifications (explore + design before editing, to prevent rework).\n"
            "DIRECT: simple, well-scoped, single change with clear scope.\n"
            "Call the tool exactly once with one decision per task.")

classify_tool = {
    "name": "classify_tasks",
    "description": "Classify each task as plan-mode or direct-execution with a one-line reason.",
    "input_schema": {"type": "object", "properties": {"decisions": {"type": "array", "items": {
        "type": "object", "properties": {
            "task_index": {"type": "integer"},
            "mode": {"type": "string", "enum": ["plan", "direct"]},
            "reason": {"type": "string"}},
        "required": ["task_index", "mode", "reason"]}}},
        "required": ["decisions"]},
}

numbered = "\n".join(f"{i}. {t}" for i, t in enumerate(TASKS))
resp = client.messages.create(
    model=MODEL, max_tokens=600, tools=[classify_tool],
    tool_choice={"type": "tool", "name": "classify_tasks"},     # force the structured decision
    messages=[{"role": "user", "content": CRITERIA + "\n\n" + numbered}])

decisions = next((b.input["decisions"] for b in resp.content if b.type == "tool_use"), [])
for d in sorted(decisions, key=lambda x: x["task_index"]):
    i = d["task_index"]
    print(f"   [{d['mode']:6}] task {i}: {TASKS[i][:48]}...")
    print(f"            -> {d['reason']}")
print("\nClaude applied the criteria — you did not hardcode the routing. The architectural / "
      "multi-file tasks land on PLAN; the single, clearly-scoped ones on DIRECT.")

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1 (sample Q5 distractor D): start DIRECT on a monolith->microservices migration
#   and "switch to plan mode only if it gets complex". The complexity is ALREADY stated in the
#   requirements (dozens of files, service boundaries) -> plan up front, or pay for rework.

# ANTI-PATTERN 2 (sample Q5 distractor B): direct execution "letting the implementation reveal
#   the service boundaries". Discovering dependencies late = exactly the costly rework plan mode
#   exists to prevent.

# ANTI-PATTERN 3: dump the whole verbose discovery into the MAIN conversation instead of an
#   Explore subagent -> context-window exhaustion during a multi-phase task.

print("Correct: architectural / multi-approach / multi-file -> PLAN MODE up front (sample Q5 = A); "
      "single well-scoped change -> DIRECT; use the Explore subagent for verbose discovery.")

### 5 · In your own code — you already wrote this

- **`exercises/02-claude-code-team/README.md:48-52`** — the plan-vs-direct note: plan mode for
  "split this monolith into microservices" (+ an Explore subagent to map the code first), direct
  for a single well-scoped change; "don't start direct and switch if it gets complex" is the
  distractor.

Read it, then re-run the cell above and confirm Claude's reasons echo those same criteria.

### 6 · Self-check (cover the answers)

1. Monolith → microservices, dozens of files, undecided service boundaries. Plan or direct? Why?
2. A single-file bug fix with a clear stack trace. Plan or direct?
3. What's wrong with "start direct, switch to plan if it gets complex"?
4. What is the Explore subagent *for*?

<details><summary>answers</summary>

1. **Plan mode** — architectural, multiple valid approaches, many files; explore/design before
   editing prevents costly rework (sample Q5 = A). 2. **Direct** — clear, well-scoped. 3. The
   complexity is *already known* from the requirements; deferring planning invites rework once
   dependencies surface late. 4. Isolating **verbose discovery** output and returning a summary,
   to preserve the main conversation's context.
</details>

## 3.5 · Iterative refinement for progressive improvement

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.5: Apply iterative refinement techniques for progressive improvement**
>
> **Knowledge of:**
> - Concrete input/output examples as the most effective way to communicate expected
>   transformations when prose descriptions are interpreted inconsistently
> - Test-driven iteration: writing test suites first, then iterating by sharing test failures to
>   guide progressive improvement
> - The interview pattern: having Claude ask questions to surface considerations the
>   developer may not have anticipated before implementing
> - When to provide all issues in a single message (interacting problems) versus fixing them
>   sequentially (independent problems)
>
> **Skills in:**
> - Providing 2-3 concrete input/output examples to clarify transformation requirements
>   when natural language descriptions produce inconsistent results
> - Writing test suites covering expected behavior, edge cases, and performance
>   requirements before implementation, then iterating by sharing test failures
> - Using the interview pattern to surface design considerations (e.g., cache invalidation
>   strategies, failure modes) before implementing solutions in unfamiliar domains
> - Providing specific test cases with example input and expected output to fix edge case
>   handling (e.g., null values in migration scripts)
> - Addressing multiple interacting issues in a single detailed message when fixes interact,
>   versus sequential iteration for independent issues

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| concrete I/O examples | When prose is read inconsistently, **2-3 input→output examples** pin the exact transform far better than more adjectives. |
| test-driven iteration | Write the tests **first**, then iterate by handing Claude the **failures** — each failure is a precise, concrete instruction. |
| interview pattern | Ask Claude to **ask you questions** before coding, surfacing edge cases (cache invalidation, failure modes) you hadn't considered. |
| batch vs sequential | **Interacting** problems → one detailed message (so fixes don't fight). **Independent** problems → fix sequentially. |

### 3 · Run it — prose vs 2 concrete examples, on the SAME inputs (real A/B)

This task statement is about a Claude *behavior*: vague prose drifts; concrete examples pin the
output. So we make **two real calls** on identical inputs — one with a vague prose instruction,
one with 2 input→output examples — and compare. The examples version is consistent and matches
the format you actually wanted; the prose version is left to interpretation.

In [ ]:
INPUTS = ["jane MARY doe", "bob smith", "a. b. cee"]

PROSE = "Format each author name properly. Return one per line, same order, nothing else."

FEWSHOT = ("Format each author name as 'Surname, F.M.' (surname capitalized, then initials with "
           "dots). Examples:\n"
           "  john ronald tolkien -> Tolkien, J.R.\n"
           "  ada LOVELACE -> Lovelace, A.\n"
           "Return one per line, same order, nothing else.")

def run_format(instruction):
    r = client.messages.create(model=MODEL, max_tokens=120,
        messages=[{"role": "user", "content": instruction + "\n\n" + "\n".join(INPUTS)}])
    return "".join(b.text for b in r.content if b.type == "text").strip()

print("INPUTS:", INPUTS)
print("\n--- PROSE-ONLY ('format properly') — interpretation is left to Claude ---")
print(run_format(PROSE))
print("\n--- 2 CONCRETE EXAMPLES (few-shot) — the transform is pinned ---")
print(run_format(FEWSHOT))
print("\nLesson: the two I/O examples communicate 'Surname, F.M.' exactly; the prose version "
      "can't, no matter how many adjectives you add. (3.5: examples > prose for transforms.)")

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1: keep ADDING PROSE ("be more careful", "format it nicely", "do it properly")
#   when output is inconsistent. Prose is read inconsistently; 2-3 I/O examples fix it directly.

# ANTI-PATTERN 2: implement first, then eyeball the result. Instead write the TEST SUITE FIRST
#   and iterate by sharing the concrete FAILURES (test-driven iteration).

# ANTI-PATTERN 3: in an unfamiliar domain, jump straight to code. Use the INTERVIEW PATTERN:
#   have Claude ask questions first (cache invalidation? failure modes?) to surface edge cases.

# ANTI-PATTERN 4: fire off INTERACTING fixes one at a time -> they fight each other. Batch
#   interacting issues into ONE detailed message; only INDEPENDENT issues go sequentially.

print("Correct: communicate transforms with 2-3 concrete I/O examples; iterate via test "
      "failures; interview-first in unfamiliar domains; batch interacting fixes, sequence "
      "independent ones.")

### 5 · In your own code — you already wrote this

- **`exercises/05-cicd-review/review.py:76-83`** — `CRITERIA` made the reviewer's instruction
  **explicit and categorical** instead of vague prose (the refinement this section is about).
- **`exercises/05-cicd-review/README.md:54-58`** — the D3.5 note: the cheapest way to improve a
  reviewer is concrete I/O examples + test-driven iteration, not more prose.
- **`claude-with-anthropic-api/app_starter/tests/test_document.py`** — a real test suite to
  iterate against (test-driven iteration: share the failures).

Open `review.py:76-83` and see prose ("be conservative") replaced by concrete categories.

### 6 · Self-check (cover the answers)

1. Claude formats a transformation inconsistently no matter how you reword the prose. Best fix?
2. What is "test-driven iteration" with Claude, concretely?
3. You're implementing in an unfamiliar domain. What pattern surfaces edge cases *before* coding?
4. Three bug fixes that interact with each other — one message or three?

<details><summary>answers</summary>

1. Provide **2-3 concrete input→output examples** (not more prose). 2. Write the **test suite
   first**, then iterate by sharing the **failures** as precise instructions. 3. The **interview
   pattern** — have Claude ask you questions first. 4. **One** detailed message — interacting
   fixes must be coordinated; only *independent* issues are fixed sequentially.
</details>

### 3b · Try it live — run the headless flags yourself

The cell above **read** the flags from `ci_review.sh` and **simulated** the structured output via
the API. The real lesson is seeing headless mode behave like CI does — in your shell, no notebook,
no interactive session:

```bash
cd "$(git rev-parse --show-toplevel)"   # any repo
# non-interactive: prints and EXITS, never waits for input (the -p that fixes the hang)
claude -p "List 2 security risks in this repo's shell scripts" --output-format json | jq .
```

- Drop the `-p` and the bare `claude "..."` would open the interactive UI and **wait for input** —
  the exact CI hang from **sample Q10**. `-p` / `--print` is the one-flag fix.
- Add **`--json-schema <file.json>`** to *constrain* the result to a shape a CI step can `jq` and
  post as inline PR comments (not just free-form JSON).
- Run EX5's actual gate to see it end-to-end (it `git diff`s, reviews with an **independent**
  `-p` instance, and `exit 1`s on a high-severity finding):
  ```bash
  cd "$(git rev-parse --show-toplevel)/ccaf-prep/exercises/05-cicd-review" && bash ci_review.sh
  ```
  Its `CLAUDE.md` feeds review context automatically, and because it's a **separate** `claude -p`
  call the reviewer never saw any author conversation — the session-isolation point (D4.6).

> `CLAUDE_HEADLESS` / `--batch` don't exist. The real trio is `-p`/`--print` +
> `--output-format json` + `--json-schema`.

## 3.6 · Integrate Claude Code into CI/CD pipelines

### 1 · What the guide actually says (verbatim)

> **Task Statement 3.6: Integrate Claude Code into CI/CD pipelines**
>
> **Knowledge of:**
> - The `-p` (or `--print`) flag for running Claude Code in non-interactive mode in automated
>   pipelines
> - `--output-format json` and `--json-schema` CLI flags for enforcing structured output
>   in CI contexts
> - CLAUDE.md as the mechanism for providing project context (testing standards, fixture
>   conventions, review criteria) to CI-invoked Claude Code
> - Session context isolation: why the same Claude session that generated code is less
>   effective at reviewing its own changes compared to an independent review instance
>
> **Skills in:**
> - Running Claude Code in CI with the `-p` flag to prevent interactive input hangs
> - Using `--output-format json` with `--json-schema` to produce machine-parseable
>   structured findings for automated posting as inline PR comments
> - Including prior review findings in context when re-running reviews after new commits,
>   instructing Claude to report only new or still-unaddressed issues to avoid duplicate
>   comments
> - Providing existing test files in context so test generation avoids suggesting duplicate
>   scenarios already covered by the test suite
> - Documenting testing standards, valuable test criteria, and available fixtures in
>   CLAUDE.md to improve test generation quality and reduce low-value test output

### 2 · Plain-English unpacking (bullet by bullet)

| Guide bullet | In plain words |
|---|---|
| `-p` / `--print` | Non-interactive mode: run the prompt, print the result, exit. **Without it, CI hangs** waiting for input. |
| `--output-format json` + `--json-schema` | Constrain the output to a **machine-parseable** shape a CI step can `jq` and act on (post comments, fail the build). |
| CLAUDE.md = CI context | The repo's `CLAUDE.md` auto-feeds testing standards / criteria / fixtures to the CI-invoked Claude — better output, less junk. |
| independent review instance | A **fresh** Claude (separate `-p` call / fresh message list) reviews better than the session that *wrote* the code — it can't rationalize its own reasoning. |
| report only new issues | On a re-run after new commits, pass **prior findings** and ask for only new/unaddressed ones → no duplicate comments. |

### 3 · Run it — the real flags + the structured artifact a CI step parses

Two observables. (a) The CI **flags** are real CLI facts — we read them from your actual
`ci_review.sh` (no paraphrase) and confirm the distractors aren't there. (b) The **structured
findings** are a real Claude behavior — an *independent* reviewer (fresh message list) is forced
to emit JSON via `tool_use`, exactly the artifact a CI step would `jq` and post.

In [ ]:
# (a) The REAL headless flags, read straight from your ci_review.sh.
ci = (EX5 / "ci_review.sh").read_text()
print("Flags present in the real ci_review.sh:")
for flag in ["-p ", "--print", "--output-format json", "--json-schema"]:
    print(f"   {'FOUND ' if flag.strip() in ci else 'MISSING':6} {flag.strip()}")
bogus = [d for d in ("CLAUDE_HEADLESS", "--batch") if d in ci]
print(f"   distractor flags present? {bogus or 'none'}  (CLAUDE_HEADLESS / --batch do NOT exist)")

# (b) The structured CI artifact — REAL call, INDEPENDENT reviewer, FORCED tool_use (mirrors review.py).
BUGGY = ("def charge(account, amount):\n"
         "    log = open('/tmp/charge.log', 'a')   # opened, never closed\n"
         "    account['balance'] -= amount\n"
         "    return True")

report_tool = {
    "name": "report_findings",
    "description": "Emit code-review findings as structured data for the CI pipeline to parse.",
    "input_schema": {"type": "object", "properties": {"findings": {"type": "array", "items": {
        "type": "object", "properties": {
            "severity": {"type": "string", "enum": ["low", "medium", "high"]},
            "file": {"type": "string"},
            "line": {"type": "integer"},
            "category": {"type": "string", "enum": ["security", "correctness", "resource-leak"]},
            "message": {"type": "string"}},
        "required": ["severity", "file", "line", "category", "message"]}}},
        "required": ["findings"]},
}
# Independent reviewer == fresh message list; it never saw any "author" conversation.
system = ("You are an INDEPENDENT code reviewer; you did not write this code and have no access "
          "to the author's reasoning. Report ONLY: security, correctness, resource-leak.")
resp = client.messages.create(
    model=MODEL, max_tokens=300, system=system, tools=[report_tool],
    tool_choice={"type": "tool", "name": "report_findings"},          # forced structured output
    messages=[{"role": "user", "content": "Review payments.py:\n" + BUGGY}])

findings = next((b.input["findings"] for b in resp.content if b.type == "tool_use"), [])
print("\nMachine-parseable artifact a CI step would consume (jq -> post as PR comments):")
print(json.dumps({"findings": findings}, indent=2))
print("\ntool_use + json schema guarantees the SHAPE (no parse errors) — not that the findings "
      "are semantically complete. The reviewer is INDEPENDENT, not the session that wrote it.")

### 4 · The anti-patterns (so you recognize the distractor)

In [ ]:
# ANTI-PATTERN 1 (sample Q10): run  claude "Analyze this PR"  in CI with no -p flag.
#   It waits for interactive input and the job HANGS forever. Add -p / --print.

# ANTI-PATTERN 2 (sample Q10 distractors): CLAUDE_HEADLESS=true env var, or a --batch flag.
#   Neither exists. The real flags are -p / --print, --output-format json, --json-schema.

# ANTI-PATTERN 3: review the code in the SAME session that generated it. Self-review is weak —
#   it rationalizes its own reasoning. Use an INDEPENDENT instance (a separate -p call).

# ANTI-PATTERN 4: re-run a review after new commits with no prior findings in context ->
#   duplicate comments. Pass prior findings; ask for only new/unaddressed issues.

print("Correct: headless CI = claude -p ... --output-format json --json-schema (sample Q10 = A); "
      "CLAUDE.md supplies project context; review with a FRESH, independent instance.")

### 5 · In your own code — you already wrote this

- **`exercises/05-cicd-review/ci_review.sh:11-27`** — the real headless invocation: `-p`,
  `--output-format json`, `--json-schema findings.schema.json` (the comment block names the
  bogus distractors `CLAUDE_HEADLESS` / `--batch`).
- **`exercises/05-cicd-review/review.py:86-111`** — `REPORT_TOOL`: structured output via
  `tool_use` + JSON schema (what CI parses).
- **`exercises/05-cicd-review/review.py:120-137`** — `_review()` runs every review in a **fresh
  message list** with an "INDEPENDENT reviewer" system prompt (the session-isolation bullet).

Open `ci_review.sh` and map each flag to a knowledge bullet above.

### 6 · Self-check (cover the answers)

1. A CI job runs `claude "review this PR"` and hangs. One-flag fix?
2. Which two flags make the output machine-parseable and schema-constrained?
3. Why use a separate `claude -p` instance to review code Claude just generated?
4. On a re-run after new commits, how do you avoid posting duplicate comments?

<details><summary>answers</summary>

1. Add **`-p`** (`--print`) for non-interactive mode (sample Q10 = A; `CLAUDE_HEADLESS` and
   `--batch` don't exist). 2. **`--output-format json`** + **`--json-schema`**. 3. **Session
   isolation** — the generating session rationalizes its own reasoning; a fresh, independent
   instance reviews more effectively. 4. Pass the **prior findings** in context and instruct
   Claude to report only **new or still-unaddressed** issues.
</details>

---
## Coming next / related domains

You've made all of Domain 3 observable on your **own** artifacts: the CLAUDE.md hierarchy +
`@import` (3.1), project commands & forked skills (3.2), path-globbed rules (3.3), the
plan-vs-direct decision applied by Claude (3.4), prose-vs-examples refinement (3.5), and the
headless CI flags + structured findings (3.6).

**Threads that continue elsewhere:**
- **D2.4 / D2.5** — `.mcp.json` config and the built-in tools (Grep/Glob/Read/Write/Edit) that
  EX2's `CLAUDE.md` tells Claude to prefer → `notebooks/D2_tool_design_mcp.ipynb`.
- **D1.6 / D4.6** — the per-file + integration **decomposition** behind EX5's reviewer, and why
  an *independent* instance beats self-review → `D1_agentic_loops.ipynb` §1.6 and Domain 4.
- **D4.1 / D4.3 / D4.5** — explicit review **criteria**, forced **structured output**, and when
  a CI gate must stay **synchronous** vs go to the Batch API → Domain 4.

Next domain to build (per the study order **D1 → D2 → D5 → D3 → D4**): **D4 — Prompt
Engineering & Structured Output (20%)**.

## Sample exam questions — Domain 3
<!-- ccaf:closing -->

Official sample questions whose tested skill lives in this domain. Cover the answer, say why
each of the other three is a distractor, then reveal. (You'll meet them again, grouped by
scenario, in `mock_exam_and_review.ipynb`.)

**Question 4.** You want to create a custom `/review` slash command that runs your team's
standard code review checklist. This command should be available to every developer when they
clone or pull the repository. Where should you create this command file?

- **A)** In the `.claude/commands/` directory in the project repository
- **B)** In `~/.claude/commands/` in each developer's home directory
- **C)** In the `CLAUDE.md` file at the project root
- **D)** In a `.claude/config.json` file with a commands array

<sub>Maps to: D3 §3.2 (custom slash commands & skills)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Project-scoped custom slash commands should be stored in the `.claude/commands/` directory
within the repository. These commands are version-controlled and automatically available to all
developers when they clone or pull the repo. Option B (`~/.claude/commands/`) is for personal
commands that aren't shared via version control. Option C (`CLAUDE.md`) is for project
instructions and context, not command definitions. Option D describes a configuration mechanism
that doesn't exist in Claude Code.

</details>

---

**Question 5.** You've been assigned to restructure the team's monolithic application into
microservices. This will involve changes across dozens of files and requires decisions about
service boundaries and module dependencies. Which approach should you take?

- **A)** Enter plan mode to explore the codebase, understand dependencies, and design an
  implementation approach before making changes.
- **B)** Start with direct execution and make changes incrementally, letting the implementation
  reveal the natural service boundaries.
- **C)** Use direct execution with comprehensive upfront instructions detailing exactly how each
  service should be structured.
- **D)** Begin in direct execution mode and only switch to plan mode if you encounter unexpected
  complexity during implementation.

<sub>Maps to: D3 §3.4 (plan mode vs direct execution)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Plan mode is designed for complex tasks involving large-scale changes, multiple valid
approaches, and architectural decisions—exactly what monolith-to-microservices restructuring
requires. It enables safe codebase exploration and design before committing to changes. Option B
risks costly rework when dependencies are discovered late. Option C assumes you already know the
right structure without exploring the code. Option D ignores that the complexity is already
stated in the requirements, not something that might emerge later.

</details>

---

**Question 6.** Your codebase has distinct areas with different coding conventions: React
components use functional style with hooks, API handlers use async/await with specific error
handling, and database models follow a repository pattern. Test files are spread throughout the
codebase alongside the code they test (e.g., `Button.test.tsx` next to `Button.tsx`), and you
want all tests to follow the same conventions regardless of location. What's the most
maintainable way to ensure Claude automatically applies the correct conventions when generating
code?

- **A)** Create rule files in `.claude/rules/` with YAML frontmatter specifying glob patterns to
  conditionally apply conventions based on file paths
- **B)** Consolidate all conventions in the root `CLAUDE.md` file under headers for each area,
  relying on Claude to infer which section applies
- **C)** Create skills in `.claude/skills/` for each code type that include the relevant
  conventions in their SKILL.md files
- **D)** Place a separate `CLAUDE.md` file in each subdirectory containing that area's specific
  conventions

<sub>Maps to: D3 §3.3 (path-specific rules)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

Option A is correct because `.claude/rules/` with glob patterns (e.g., `**/*.test.tsx`) allows
conventions to be automatically applied based on file paths regardless of directory
location—essential for test files spread throughout the codebase. Option B relies on inference
rather than explicit matching, making it unreliable. Option C requires manual skill invocation or
relies on Claude choosing to load them, contradicting the need for deterministic "automatic"
application based on file paths. Option D can't easily handle files spread across many
directories since CLAUDE.md files are directory-bound.

</details>

---

**Question 10.** Your pipeline script runs `claude "Analyze this pull request for security
issues"` but the job hangs indefinitely. Logs indicate Claude Code is waiting for interactive
input. What's the correct approach to run Claude Code in an automated pipeline?

- **A)** Add the `-p` flag: `claude -p "Analyze this pull request for security issues"`
- **B)** Set the environment variable `CLAUDE_HEADLESS=true` before running the command
- **C)** Redirect stdin from `/dev/null`: `claude "Analyze this pull request for security issues" < /dev/null`
- **D)** Add the `--batch` flag: `claude --batch "Analyze this pull request for security issues"`

<sub>Maps to: D3 §3.6 (CI/CD integration)</sub>

<details><summary>Answer + explanation</summary>

**Correct answer: A**

The `-p` (or `--print`) flag is the documented way to run Claude Code in non-interactive mode. It
processes the prompt, outputs the result to stdout, and exits without waiting for user
input—exactly what CI/CD pipelines require. The other options reference non-existent features
(`CLAUDE_HEADLESS` environment variable, `--batch` flag) or use Unix workarounds that don't
properly address Claude Code's command syntax.

</details>

## Exercises that use this domain

Exercises are **cross-domain** — none is D3-only — so do each once all its domains are studied
(study order **D1 → D2 → D5 → D3 → D4**; see [`README.md`](./README.md) for the wave order).

| Exercise | Domains it reinforces | Do it after you've studied |
|----------|-----------------------|-----------------------------|
| **Ex2** `../exercises/02-claude-code-team/`  | D3 + D2      | D2, **D3** ✅ (this notebook) |
| **Ex5** `../exercises/05-cicd-review/`       | D3 + D4      | D3, **D4** |

Finishing D3 **unlocks Ex2** (its other domain, D2, is already done) — that's the EX2 team-repo
config you parsed all through this notebook. **Ex5** waits on **D4** (it also leans on explicit
criteria, structured output, and batch-vs-sync — the next domain).